# Inflation curves (CPI / breakeven and real rates)

Deep-dive reference: dedicated `InflationCurve` bindings for CPI paths, observation lags, and inflation-linked pricing intuition.

## Concept

**Inflation curves** describe expected **CPI** growth or **inflation swap** breakevens, used to price **TIPS**, **ILBs**, and **inflation swaps**. A common identity (Fisher-style, continuous-time shorthand) links **nominal** \(r_n\), **real** \(r_r\), and **inflation** \(\pi\): \(r_n \approx r_r + \pi\). When no dedicated curve type exists, you can still illustrate **real zero rates** from a **nominal** curve and an assumed **inflation** path.

## API walkthrough

`InflationCurve` is available directly from `finstack_quant.core.market_data`. Build it from CPI knot points, query raw CPI levels with `cpi(t)`, apply the contractual observation lag with `cpi_with_lag(t)`, read the linker uplift factor with `index_ratio(t)`, and compute annualized inflation between horizons with `inflation_rate(t1, t2)`.

In [1]:
import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF

from finstack_quant.core.market_data import InflationCurve

curve = InflationCurve(
    "USD-CPI",
    DEMO_AS_OF,
    315.5,
    [(0.0, 315.5), (1.0, 323.4), (2.0, 331.8), (5.0, 357.9), (10.0, 405.4)],
    indexation_lag_months=3,
    interp="log_linear",
)
print("curve:", curve)
print("base_cpi:", curve.base_cpi, "lag_months:", curve.indexation_lag_months)
for t in (1.0, 2.0, 5.0):
    print(f"t={t}y  cpi={curve.cpi(t):.3f}  cpi_with_lag={curve.cpi_with_lag(t):.3f}")
print("5Y annualized inflation:", round(curve.inflation_rate(0.0, 5.0), 6))
print("5Y simple (non-compounded) inflation:", round(curve.inflation_rate_simple(0.0, 5.0), 6))


curve: InflationCurve(id="USD-CPI")
base_cpi: 315.5 lag_months: 3
t=1.0y  cpi=323.400  cpi_with_lag=321.407
t=2.0y  cpi=331.800  cpi_with_lag=329.680
t=5.0y  cpi=357.900  cpi_with_lag=355.649
5Y annualized inflation: 0.02554
5Y simple (non-compounded) inflation: 0.026878


## Practical example

**Nominal OIS** discount curve plus a **constant CPI breakeven** (illustrative) to derive **approximate real zeros** with \(r_r \approx r_n - \pi\).

**Caveats:** Real **CPI** is **lagged** (release delay and contract **observation lags**), **seasonal** (and often revised), and market breakevens embed an **inflation risk premium** and liquidity effects—so a flat \(\pi\) is only a teaching shortcut. In **continuous compounding**, if nominal and real zeros are defined consistently, \(DF(t) \approx e^{-r t}\) links discounting to zeros; a Fisher-style split \(r_n \approx r_r + \pi\) is still an approximation and must use compatible conventions in production.

In [2]:
# `index_ratio(t)` is the curve-level accessor for the linker uplift factor.
# It is the same quantity the valuations layer reports per cashflow as
# `inflation_index_ratio`, so reading it off the curve keeps this walkthrough
# on the pricing path rather than re-deriving cpi_with_lag(t) / base_cpi here.
notional = 1_000_000.0
for t in (1.0, 3.0, 5.0, 10.0):
    ratio = curve.index_ratio(t)
    adjusted_principal = notional * ratio
    print(
        f"t={t:>4.1f}y  lagged_index_ratio={ratio:.6f}  inflation_adjusted_notional={adjusted_principal:,.2f}"
    )

t= 1.0y  lagged_index_ratio=1.018722  inflation_adjusted_notional=1,018,721.54
t= 3.0y  lagged_index_ratio=1.071762  inflation_adjusted_notional=1,071,761.98
t= 5.0y  lagged_index_ratio=1.127254  inflation_adjusted_notional=1,127,254.28
t=10.0y  lagged_index_ratio=1.276963  inflation_adjusted_notional=1,276,962.90


## Takeaways

- `InflationCurve` gives you direct access to CPI levels, lag-adjusted CPI, and annualized inflation over arbitrary horizons.
- Observation lag matters: `cpi_with_lag(t)` is the right conceptual hook for linker and inflation-swap settlement mechanics.
- `index_ratio(t)` returns the principal uplift factor for an inflation-linked security. It is the curve-level view of the `inflation_index_ratio` the valuations layer stamps on each cashflow, so notebook and pricer agree by construction rather than by convention. Note it applies **no deflation floor** — real linkers usually do.
- Calibrated inflation curves still depend on discounting, seasonality choices, and instrument conventions, so treat the simple examples here as structure-first rather than full production setup.

## Calibration from inflation swap quotes

With the calibration engine, we can bootstrap an **inflation curve** from zero-coupon inflation swap (ZCIS) rates.  This requires:
1. A pre-existing **discount curve** (for PV calculations)
2. ZCIS quotes at various maturities with a **base CPI** level and **observation lag**

The engine solves for the CPI projection path that reprices each ZCIS to zero NPV.

In [3]:
import json
from pathlib import Path

from finstack_quant.valuations import calibrate

_NOTEBOOK_DATA = json.loads(Path("data/inflation_curves.json").read_text())
BASE_DATE = DEMO_AS_OF.isoformat()


def tenor(count, unit):
    return {"tenor": {"count": count, "unit": unit}}


rate_quotes = [
    {"kind": "rate_quote", "type": "deposit", "id": "USD-SOFR-DEP-1M", "index": "USD-SOFR-OIS", "pillar": tenor(1, "months"), "rate": 0.0525},
    {"kind": "rate_quote", "type": "deposit", "id": "USD-SOFR-DEP-3M", "index": "USD-SOFR-OIS", "pillar": tenor(3, "months"), "rate": 0.0520},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-1Y", "index": "USD-SOFR-OIS", "pillar": tenor(1, "years"), "rate": 0.0510},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-2Y", "index": "USD-SOFR-OIS", "pillar": tenor(2, "years"), "rate": 0.0490},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-5Y", "index": "USD-SOFR-OIS", "pillar": tenor(5, "years"), "rate": 0.0450},
    {"kind": "rate_quote", "type": "swap", "id": "USD-OIS-SWAP-10Y", "index": "USD-SOFR-OIS", "pillar": tenor(10, "years"), "rate": 0.0410},
]

inflation_quotes = _NOTEBOOK_DATA['inflation_quotes']

plan = {
    "schema": "finstack_quant.calibration",
    "plan": {
        "id": "us-cpi-curve",
        "description": "US CPI inflation curve from ZCIS quotes",
        "quote_sets": {
            "ois": [q["id"] for q in rate_quotes],
            "zcis": [q["inflation_swap"]["id"] for q in inflation_quotes],
        },
        "steps": [
            {
                "id": "USD-OIS",
                "quote_set": "ois",
                "kind": "discount",
                "curve_id": "USD-OIS",
                "currency": "USD",
                "base_date": BASE_DATE,
                "method": "Bootstrap",
                "interpolation": "linear",
                "extrapolation": "flat_forward",
            },
            {
                "id": "USD-CPI",
                "quote_set": "zcis",
                "kind": "inflation",
                "curve_id": "USD-CPI",
                "currency": "USD",
                "base_date": BASE_DATE,
                "discount_curve_id": "USD-OIS",
                "index": "USA-CPI-U",
                "observation_lag": "3M",
                "base_cpi": 315.5,
                "method": "Bootstrap",
                "interpolation": "linear",
            },
        ],
        "settings": {"use_parallel": False},
    },
    "market_data": rate_quotes + inflation_quotes,
}

result = calibrate(json.dumps(plan))
print("Success:", result.success)
print(result.report_to_dataframe().to_string(index=False))
print()

cal_curve = result.market.get_inflation_curve("USD-CPI")
print("Calibrated CPI path:")
for t in (1.0, 3.0, 5.0, 10.0):
    print(
        f"  t={t:>4.1f}y  cpi={cal_curve.cpi(t):.3f}"
        f"  annualized_inflation={cal_curve.inflation_rate(0.0, t):.6f}"
        f"  index_ratio={cal_curve.index_ratio(t):.6f}"
    )

Success: True
step_id  success  iterations  max_residual         rmse                                                                          convergence_reason
USD-CPI     True         298  1.611489e-13 1.332683e-13 generic bootstrap calibration converged: max_residual (1.61e-13) within tolerance (1.00e-8)
USD-OIS     True         475  8.161587e-13 5.484864e-13 generic bootstrap calibration converged: max_residual (8.16e-13) within tolerance (1.00e-8)

Calibrated CPI path:
  t= 1.0y  cpi=325.719  annualized_inflation=0.032390  index_ratio=1.025409
  t= 3.0y  cpi=343.304  annualized_inflation=0.028553  index_ratio=1.081258
  t= 5.0y  cpi=360.716  annualized_inflation=0.027148  index_ratio=1.136216
  t=10.0y  cpi=403.309  annualized_inflation=0.024858  index_ratio=1.278218
